# Seminar 2. Custom PyTorch Operators

# Building Models in PyTorch Through Composition

PyTorch models are built using **composition**.  
Instead of defining one large monolithic network, we construct models by combining smaller, reusable modules.

Each module can contain other modules, which allows us to build hierarchical and well-structured architectures.

---

## Composition

Composition means:

- A model is built from smaller blocks.
- Each block can contain multiple layers.
- Blocks can be reused in larger architectures.
- Complex models are created by stacking simpler components.

This keeps code:

- Modular  
- Reusable  
- Readable  
- Easy to extend  


## Key Ideas

- Inherit from `nn.Module`
- Define layers inside `__init__`
- Define computation in `forward()`
- Create reusable blocks
- Build larger models by combining blocks

---

## Example: Model Built from Two Blocks

Below is a simple example where:

- We define a reusable blocks: `LinearReLUBlock` and `LinearTanhBlock`
- The final model is composed of two such blocks


In [1]:
import torch
import torch.nn as nn


class LinearReLUBlock(nn.Module):
    def __init__(self, in_features: int, out_features: int):
        super().__init__()

        self.linear = nn.Linear(in_features, out_features)
        self.activation = nn.ReLU()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.linear(x)
        x = self.activation(x)
        return x


class LinearTanhBlock(nn.Module):
    def __init__(self, in_features: int, out_features: int):
        super().__init__()

        self.linear = nn.Linear(in_features, out_features)
        self.activation = nn.Tanh()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.linear(x)
        x = self.activation(x)
        return x


class CombinedModel(nn.Module):
    def __init__(self):
        super().__init__()

        self.block1 = LinearReLUBlock(4, 8)
        self.block2 = LinearTanhBlock(8, 8)
        self.output = nn.Linear(8, 2)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.block1(x)
        x = self.block2(x)
        x = self.output(x)
        return x


model = CombinedModel()
print(model)

CombinedModel(
  (block1): LinearReLUBlock(
    (linear): Linear(in_features=4, out_features=8, bias=True)
    (activation): ReLU()
  )
  (block2): LinearTanhBlock(
    (linear): Linear(in_features=8, out_features=8, bias=True)
    (activation): Tanh()
  )
  (output): Linear(in_features=8, out_features=2, bias=True)
)


# What `nn.Module` Enables

When we inherit from `nn.Module`, we automatically gain powerful functionality that works **recursively across all submodules**.

## What `nn.Module` Gives Us



### Parameter Registration

All layers assigned as attributes (e.g. `self.linear = nn.Linear(...)`) are:

- Automatically registered
- Collected in `model.parameters()`
- Included in `model.state_dict()`

This works **recursively** for all sub-blocks.

In [2]:
print("Registered parameters:")
for name, param in model.named_parameters():
    print(name, param.shape)


Registered parameters:
block1.linear.weight torch.Size([8, 4])
block1.linear.bias torch.Size([8])
block2.linear.weight torch.Size([8, 8])
block2.linear.bias torch.Size([8])
output.weight torch.Size([2, 8])
output.bias torch.Size([2])


### Automatic Gradient Tracking

During the forward pass:

- PyTorch dynamically builds a computation graph
- Calling `loss.backward()` computes gradients
- Gradients are stored in each parameter’s `.grad`

No manual graph management is required.

In [3]:
x = torch.randn(5, 4)
target = torch.randn(5, 2)

criterion = nn.MSELoss()
output = model(x)
loss = criterion(output, target)

loss.backward()

print("\nGradient computed for output layer:",
      model.output.weight.grad is not None)
model.output.weight.grad


Gradient computed for output layer: True


tensor([[ 0.0123, -0.0499,  0.0841,  0.0665, -0.0385,  0.0660, -0.0112,  0.1811],
        [-0.0766, -0.0203, -0.0185, -0.0658,  0.0231,  0.0232,  0.0202, -0.0270]])

### Device and Type Transfer (`.to()`)

Calling:

    model.to(device)

or

    model.to(dtype)

moves all:

- Parameters
- Buffers
- Submodules

to CPU/GPU/dtype automatically.

In [4]:
import torch

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

dtype = torch.float32

model = model.to(device=device, dtype=dtype)

x: torch.Tensor = torch.randn(5, 4, device=device, dtype=dtype)

print("Model device:", next(model.parameters()).device)
print("Model dtype:", next(model.parameters()).dtype)

Model device: cpu
Model dtype: torch.float32


### Saving & Loading (`state_dict()`)

- `model.state_dict()` returns all parameters recursively
- `model.load_state_dict(...)` restores them

This works across the full module tree.

In [5]:
state_dict = model.state_dict()
torch.save(state_dict, "combined_model.pt")

# `train()` vs `eval()` Mode in PyTorch

PyTorch modules have two main modes: **training mode** and **evaluation mode**.  
Switching between them affects layers that behave differently during training and inference.

---

## `model.train()`

- Sets the model to **training mode**.
- Used when training the model with gradient updates.
- Affects certain layers, such as:

| Layer Type        | Behavior in `train()` Mode                  |
|------------------|--------------------------------------------|
| `Dropout`         | Randomly zeroes some activations           |
| `BatchNorm`       | Updates running statistics (mean/variance) |

- Gradients are computed as usual.

---

## `model.eval()`

- Sets the model to **evaluation (inference) mode**.
- Used when evaluating or deploying the model.
- Affects certain layers:

| Layer Type        | Behavior in `eval()` Mode                   |
|------------------|--------------------------------------------|
| `Dropout`         | Passes all activations through unchanged  |
| `BatchNorm`       | Uses stored running mean/variance         |

- No layers update internal statistics.
- Gradients are usually not required (often used with `torch.no_grad()`).

---

## Key Points

- Always use `model.train()` during training.
- Always use `model.eval()` during evaluation or testing.
- Forgetting to switch can lead to inconsistent results, especially with `Dropout` or `BatchNorm`.



In [6]:
import torch
import torch.nn as nn

# Simple model with Dropout and BatchNorm
class SimpleModel(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.fc1 = nn.Linear(4, 8)
        self.bn = nn.BatchNorm1d(8)
        self.dropout = nn.Dropout(p=0.5)
        self.fc2 = nn.Linear(8, 2)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.fc1(x)
        x = self.bn(x)
        x = torch.relu(x)
        x = self.dropout(x)
        x = self.fc2(x)
        return x


model = SimpleModel()
x = torch.randn(5, 4)

# Training mode
model.train()
output_train = model(x)
print("Training mode output:\n", output_train)

# Evaluation mode
model.eval()
with torch.no_grad():
    output_eval = model(x)
print("Evaluation mode output:\n", output_eval)


Training mode output:
 tensor([[-0.3433,  0.0309],
        [-1.3435,  0.0567],
        [-0.6032,  0.3733],
        [-0.3406,  0.0275],
        [-1.7608,  0.4676]], grad_fn=<AddmmBackward0>)
Evaluation mode output:
 tensor([[-0.5293,  0.0065],
        [-0.9239,  0.2167],
        [-0.5824,  0.2234],
        [-0.8030, -0.1388],
        [-0.9813,  0.2738]])


# `torch.no_grad()` and `torch.inference_mode()` in PyTorch

When performing inference (evaluating a model without updating parameters), PyTorch provides context managers to **disable gradient tracking**. This saves memory and speeds up computation.

---

## `torch.no_grad()`

- Disables gradient tracking.
- Useful during evaluation or inference.
- Gradients are **not computed**, but autograd still tracks operations for some internal purposes.
- Can be used as a **context manager** or a **function decorator**.

---

## `torch.inference_mode()`

- Introduced in PyTorch 1.9.
- Similar to `no_grad()`, but **more efficient**.
- Completely disables autograd and reduces memory usage.
- Recommended for pure inference pipelines.



In [7]:
import torch
import torch.nn as nn

class SimpleModel(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.fc = nn.Linear(4, 2)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.fc(x)

    # -----------------------------
    # Using torch.no_grad() as method decorator
    # -----------------------------
    @torch.no_grad()
    def forward_no_grad(self, x: torch.Tensor) -> torch.Tensor:
        return self.fc(x)

    # -----------------------------
    # Using torch.inference_mode() as method decorator
    # -----------------------------
    @torch.inference_mode()
    def forward_inference(self, x: torch.Tensor) -> torch.Tensor:
        return self.fc(x)


model = SimpleModel()
x = torch.randn(5, 4)

# -----------------------------
# Call decorated methods
# -----------------------------
output_no_grad_method = model.forward_no_grad(x)
output_infer_method = model.forward_inference(x)

print("Output no_grad method:\n", output_no_grad_method)
print("Output inference_mode method:\n", output_infer_method)

# -----------------------------
# Using context managers
# -----------------------------

with torch.no_grad():
    output_no_grad_cm = model(x)

with torch.inference_mode():
    output_infer_cm = model(x)

print("Output no_grad context manager:\n", output_no_grad_cm)
print("Output inference_mode context manager:\n", output_infer_cm)


Output no_grad method:
 tensor([[-8.2876e-02,  5.9021e-04],
        [ 5.1865e-02, -2.9307e-01],
        [-2.7057e-01, -8.7899e-01],
        [ 1.7902e-02, -1.4825e-01],
        [-3.2483e-02, -5.6228e-01]])
Output inference_mode method:
 tensor([[-8.2876e-02,  5.9021e-04],
        [ 5.1865e-02, -2.9307e-01],
        [-2.7057e-01, -8.7899e-01],
        [ 1.7902e-02, -1.4825e-01],
        [-3.2483e-02, -5.6228e-01]])
Output no_grad context manager:
 tensor([[-8.2876e-02,  5.9021e-04],
        [ 5.1865e-02, -2.9307e-01],
        [-2.7057e-01, -8.7899e-01],
        [ 1.7902e-02, -1.4825e-01],
        [-3.2483e-02, -5.6228e-01]])
Output inference_mode context manager:
 tensor([[-8.2876e-02,  5.9021e-04],
        [ 5.1865e-02, -2.9307e-01],
        [-2.7057e-01, -8.7899e-01],
        [ 1.7902e-02, -1.4825e-01],
        [-3.2483e-02, -5.6228e-01]])


# Disabling Gradients with `requires_grad_(False)`

PyTorch provides a convenient method `requires_grad_()` that can **enable or disable gradients in-place** for all parameters of a model or a tensor.

Using:

```python
param.requires_grad_(False)
```

- Sets `requires_grad=False` **in-place** for that parameter.
- This is useful for freezing models during inference or transfer learning.
- Can be applied to an entire model recursively by iterating over its parameters.


In [8]:
model = SimpleModel()

# Disable gradient computation for all parameters using requires_grad_()
for param in model.parameters():
    param.requires_grad_(False)

# Verify
for name, param in model.named_parameters():
    print(f"{name}: requires_grad={param.requires_grad}")

# Forward pass still works
x = torch.randn(5, 4)
output = model(x)
print("Output shape:", output.shape)

fc.weight: requires_grad=False
fc.bias: requires_grad=False
Output shape: torch.Size([5, 2])


# Redefining `train()` and `eval()` in `nn.Module`

PyTorch’s `nn.Module` provides built-in `train(mode: bool = True)` and `eval()` methods to switch between **training** and **evaluation** modes.  

Sometimes, when creating **custom modules or blocks**, you might want to **override these methods** to perform extra actions whenever the mode changes.

---

## Why Override?

- Apply mode-specific logic to sub-blocks or attributes that are not standard layers
- Log or track mode switches
- Automatically modify internal flags or buffers along with training/eval mode

---

## How It Works

- `train(mode: bool = True)` sets `self.training = mode` for the module
- `eval()` is equivalent to `train(False)`
- Default implementation recursively calls `train(mode)` on all submodules
- Overriding allows custom behavior while keeping recursive updates intact


In [9]:
import torch
import torch.nn as nn
from torch import Tensor
from typing import Self

class CustomBlock(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.linear = nn.Linear(4, 4)
        self.dropout = nn.Dropout(p=0.5)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.linear(x)
        x = torch.relu(x)
        x = self.dropout(x)
        return x

    # -----------------------------
    # Override train() method
    # -----------------------------
    def train(self, mode: bool = True) -> Self:
        print(f"CustomBlock set to {'train' if mode else 'eval'} mode")
        super().train(mode)  # Call original method to update submodules
        # Add any custom logic here
        return self

    # -----------------------------
    # Override eval() method
    # -----------------------------
    def eval(self) -> Self:
        print("CustomBlock set to eval mode")
        return super().eval()


# Example usage
model = CustomBlock()
x = torch.randn(2, 4)

# Switch to training mode
model.train()
output_train = model(x)

# Switch to evaluation mode
model.eval()
with torch.no_grad():
    output_eval = model(x)

print("Output training mode:", output_train)
print("Output eval mode:", output_eval)


CustomBlock set to train mode
CustomBlock set to eval mode
CustomBlock set to eval mode
Output training mode: tensor([[0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.2634, 0.0000]], grad_fn=<MulBackward0>)
Output eval mode: tensor([[0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.2091, 0.1317, 0.2921]])


# Common Module Aggregators in PyTorch

When building neural networks, it is often useful to group multiple layers or submodules together.  
PyTorch provides several **module aggregators** that help organize layers and blocks. The most common ones are:



## `nn.Sequential`

- Holds modules in a sequential order.
- Executes them **in the order they are added** during the forward pass.
- Ideal for simple **stacked layers** with a single input and output.

**Key points:**

- Forward pass is automatically defined.
- Cannot handle multiple inputs or branching.

In [10]:
seq_model = nn.Sequential(
    nn.Linear(4, 8),
    nn.ReLU(),
    nn.Linear(8, 2)
)

x = torch.randn(5, 4)
output_seq = seq_model(x)
print("nn.Sequential output shape:", output_seq.shape)


nn.Sequential output shape: torch.Size([5, 2])


## `nn.ModuleList`

- Holds a **list of modules**.
- Does **not define a forward pass automatically**.
- Useful when you need to **loop over modules**, or have conditional computation.

**Key points:**

- Modules are registered properly, so parameters are tracked.
- You must define your own `forward()`.

In [11]:
class ModuleListModel(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.layers = nn.ModuleList([
            nn.Linear(4, 8),
            nn.ReLU(),
            nn.Linear(8, 2)
        ])

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        for layer in self.layers:
            x = layer(x)
        return x

ml_model = ModuleListModel()
output_ml = ml_model(x)
print("nn.ModuleList output shape:", output_ml.shape)

nn.ModuleList output shape: torch.Size([5, 2])


## `nn.ModuleDict`

- Holds modules in a **dictionary** with string keys.
- Useful for architectures with **named branches**, **dynamic selection**, or **multi-head outputs**.
- Like `ModuleList`, it does **not define a forward pass**.

In [12]:
class ModuleDictModel(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.branches = nn.ModuleDict({
            "branch1": nn.Linear(4, 8),
            "branch2": nn.Linear(4, 8)
        })
        self.output: nn.Linear = nn.Linear(8, 2)

    def forward(self, x: torch.Tensor, branch_name: str = "branch1") -> torch.Tensor:
        x = self.branches[branch_name](x)
        return self.output(x)

md_model = ModuleDictModel()
output_md = md_model(x, branch_name="branch2")
print("nn.ModuleDict output shape:", output_md.shape)

nn.ModuleDict output shape: torch.Size([5, 2])


## Type hints and Shape Information

For production code, a practical pattern is to use `Tensor` for type hints and document the expected shape next to the signature.

`Tensor` tells us what kind of object is expected, while `[B, C, H, W]` documents the tensor dimensions explicitly.


In [13]:
from torch import Tensor

def normalize_images(x: Tensor) -> Tensor:
    # x: [B, C, H, W]
    return x / 255.0



## Работа на семинаре

### LSTM

![lstm](assets/LSTM.png)

In [ ]:
import torch
from torch import nn


class LSTMCell(nn.Module):
    """One recurrent step: x [B, input_size], h and c [B, hidden_size]."""

    def __init__(self, input_size: int, hidden_size: int):
        super().__init__()
        self.input_proj = nn.Linear(input_size, 4 * hidden_size)
        self.hidden_proj = nn.Linear(hidden_size, 4 * hidden_size)

    def forward(self, x: torch.Tensor, state: tuple) -> tuple:
        h_prev, c_prev = state
        gates = self.input_proj(x) + self.hidden_proj(h_prev)
        i, f, g, o = gates.chunk(4, dim=-1)
        i, f, o = torch.sigmoid(i), torch.sigmoid(f), torch.sigmoid(o)
        g = torch.tanh(g)
        c = f * c_prev + i * g
        h = o * torch.tanh(c)
        return h, c

LSTMCell: x (2, 8), h_prev/c_prev (2, 16) -> h (2, 16), c (2, 16)
Shape and backward: OK


### Inception

![inception](assets/inception.png)

In [ ]:
class InceptionBlock(nn.Module):
    def __init__(self, in_channels: int, channels_1x1: int,
                 reduce_3x3: int, channels_3x3: int,
                 reduce_5x5: int, channels_5x5: int, pool_channels: int):
        super().__init__()
        self.branch1 = nn.Sequential(
            nn.Conv2d(in_channels, channels_1x1, kernel_size=1), nn.ReLU(),
        )
        self.branch3 = nn.Sequential(
            nn.Conv2d(in_channels, reduce_3x3, kernel_size=1), nn.ReLU(),
            nn.Conv2d(reduce_3x3, channels_3x3, kernel_size=3, padding=1), nn.ReLU(),
        )
        self.branch5 = nn.Sequential(
            nn.Conv2d(in_channels, reduce_5x5, kernel_size=1), nn.ReLU(),
            nn.Conv2d(reduce_5x5, channels_5x5, kernel_size=5, padding=2), nn.ReLU(),
        )
        self.branch_pool = nn.Sequential(
            nn.MaxPool2d(kernel_size=3, stride=1, padding=1),
            nn.Conv2d(in_channels, pool_channels, kernel_size=1), nn.ReLU(),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # All branches preserve H, W; concatenate along channels.
        return torch.cat([
            self.branch1(x), self.branch3(x), self.branch5(x), self.branch_pool(x),
        ], dim=1)


InceptionBlock: (2, 8, 16, 16) -> (2, 20, 16, 16)
Shape and backward: OK


### SE

![se](assets/SqueezeAndExcite.png)

In [ ]:
class SEBlock(nn.Module):
    """Channel recalibration for an already computed feature map U."""

    def __init__(self, channels: int, reduction: int = 16):
        super().__init__()
        if reduction <= 0:
            raise ValueError('reduction must be positive')
        hidden = max(1, channels // reduction)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.excitation = nn.Sequential(
            nn.Linear(channels, hidden), nn.ReLU(),
            nn.Linear(hidden, channels), nn.Sigmoid(),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        summary = self.pool(x).flatten(1)  # [B, C]
        weights = self.excitation(summary).unsqueeze(-1).unsqueeze(-1)
        return x * weights  # [B, C, H, W]

SEBlock: (2, 8, 16, 16) -> (2, 8, 16, 16)
Shape and backward: OK


### Selective Kernel

![selective](assets/SelectiveKernel.png)


In [ ]:
class SelectiveKernel(nn.Module):
    def __init__(self, channels: int, reduction: int = 16):
        super().__init__()
        hidden = max(1, channels // reduction)
        self.branches = nn.ModuleList([
            nn.Sequential(
                nn.Conv2d(channels, channels, kernel_size=k, padding=k // 2, bias=False),
                nn.BatchNorm2d(channels), nn.ReLU(),
            ) for k in (3, 5)
        ])
        self.fuse = nn.Sequential(nn.Linear(channels, hidden), nn.ReLU())
        self.select = nn.ModuleList([nn.Linear(hidden, channels) for _ in range(2)])

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        features = torch.stack([branch(x) for branch in self.branches], dim=1)
        summary = features.sum(dim=1).mean(dim=(-2, -1))
        z = self.fuse(summary)
        logits = torch.stack([head(z) for head in self.select], dim=1)
        weights = torch.softmax(logits, dim=1)  # Sum over branches equals 1.
        return (features * weights[..., None, None]).sum(dim=1)

SelectiveKernel: (2, 8, 16, 16) -> (2, 8, 16, 16)
Shape and backward: OK


### PatchMerger

![patchmerger](assets/PatchMerger.png)


In [ ]:
class PatchMerger(nn.Module):
    def __init__(self, dim: int, num_output_tokens: int):
        super().__init__()
        self.norm = nn.LayerNorm(dim)
        self.queries = nn.Parameter(torch.randn(num_output_tokens, dim) * 0.02)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.norm(x)
        scores = x @ self.queries.transpose(0, 1)
        weights = torch.softmax(scores.transpose(1, 2), dim=-1)
        return weights @ x

PatchMerger: (2, 12, 16) -> (2, 4, 16)
Shape and backward: OK


## Домашнее задание

## Задание 1, реализуйте требуемые блоки

## ResNet Block (0.1 балл)

![Resnet](assets/ResBlock.png)

https://arxiv.org/pdf/1512.03385

In [10]:
class ResidualBlock(nn.Module):
    def __init__(self, channels: int):
        super().__init__()

        self.layers = nn.Sequential(
            nn.Conv2d(channels, channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(channels),
            nn.ReLU(),
            nn.Conv2d(channels, channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(channels),
        )
        self.activation = nn.ReLU()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.activation(x + self.layers(x))


torch.manual_seed(42)
block = ResidualBlock(channels=8)
x = torch.randn(2, 8, 16, 16, requires_grad=True)
y = block(x)
print(f'ResidualBlock: {tuple(x.shape)} to {tuple(y.shape)}')
loss = y.square().mean()
inputs = (x,)
loss.backward()
print('Backward done')


ResidualBlock: (2, 8, 16, 16) to (2, 8, 16, 16)
Backward done


## Depthwise Separable Convolution (0.1 балл)
![DepthWiseConv](assets/DepthWiseConv.png)

https://arxiv.org/pdf/1610.02357

In [ ]:
class SeparableConv2d(nn.Module):
    def __init__(self, in_channels: int, out_channels: int,
                 kernel_size: int = 3, stride: int = 1, padding: int = 1):
        super().__init__()

        self.depthwiseConv = nn.Conv2d(
            in_channels, in_channels, kernel_size=kernel_size,
            stride=stride, padding=padding, groups=in_channels, bias=False,
        )
        self.pointwiseConv = nn.Conv2d(in_channels, out_channels, kernel_size=1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.depthwiseConv(x)
        x = self.pointwiseConv(x)
        return x


block = SeparableConv2d(in_channels=3, out_channels=8)
x = torch.randn(2, 3, 16, 16, requires_grad=True)
y = block(x)
print(f'SeparableConv2d: {tuple(x.shape)} to {tuple(y.shape)}')
loss = y.square().mean()
inputs = (x,)
assert torch.isfinite(loss)
loss.backward()
print('Backward done')


SeparableConv2d: (2, 3, 16, 16) to (2, 8, 16, 16)
Backward done


## Vanilla Attention (0.1 балл)

Let:

$$
\text{query} \in \mathbb{R}^{B \times d} \\
\text{key} \in \mathbb{R}^{B \times L \times d}
$$

---

### Alignment Scores

$$
\text{score} = \text{key} \cdot (W_\text{align} \, \text{query})^T \\
\text{score} \in \mathbb{R}^{B \times L}
$$

---

### Attention Weights

$$
\text{att} = \text{softmax}(\text{score}, \text{dim}=1) \\
\text{att} \in \mathbb{R}^{B \times L}
$$

---

### Context Vector

$$
\text{context} = \sum_{i=1}^{L} \text{att}_i \cdot \text{key}_i \\
\text{context} \in \mathbb{R}^{B \times d}
$$

---

### Output

$$
\text{out} = \tanh(W_\text{value} \, \text{context} + W_\text{query} \, \text{query}) \\
\text{out} \in \mathbb{R}^{B \times d}
$$



https://arxiv.org/abs/1409.0473


https://arxiv.org/abs/1508.04025

In [ ]:
import torch
from torch import nn

class VanillaAttention(nn.Module):
    def __init__(self, dim: int):
        super().__init__()

        self.align = nn.Linear(dim, dim, bias=False)
        self.value = nn.Linear(dim, dim, bias=False)
        self.query = nn.Linear(dim, dim, bias=False)

    def forward(self, query: torch.Tensor, key: torch.Tensor) -> torch.Tensor:
        aligned_query = self.align(query)
        scores = torch.bmm(key, aligned_query.unsqueeze(-1)).squeeze(-1)
        attention = torch.softmax(scores, dim=1)
        context = torch.bmm(attention.unsqueeze(1), key).squeeze(1)
        att = torch.tanh(self.value(context) + self.query(query))
        return att



block = VanillaAttention(dim=16)
query = torch.randn(2, 16, requires_grad=True)
key = torch.randn(2, 7, 16, requires_grad=True)
y = block(query, key)
print(f'VanillaAttention: query {tuple(query.shape)}, key {tuple(key.shape)} to {tuple(y.shape)}')
loss = y.square().mean()
inputs = (query, key)
loss.backward()
print('Backward done')


VanillaAttention: query (2, 16), key (2, 7, 16) to (2, 16)
Backward done


## Dot Product Attention (0.1 балл)

$$
Q \in \mathbb{R}^{B \times L_q \times d_k} \\
K \in \mathbb{R}^{B \times L_k \times d_k} \\
V \in \mathbb{R}^{B \times L_k \times d_k}
$$

$$
S = \frac{Q K^T}{\sqrt{d_k}}
$$

$$
\text{Attention}(Q, K, V) = \text{softmax}(S, \text{dim}=-1) \, V
$$



https://arxiv.org/abs/1706.03762


In [13]:
class ScaledDotProductAttention(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, query: torch.Tensor, key: torch.Tensor,
                value: torch.Tensor) -> torch.Tensor:
        scores = query @ key.transpose(-2, -1) / (query.shape[-1] ** 0.5)
        logits = torch.softmax(scores, dim=-1)
        attention = logits @ value
        return attention


block = ScaledDotProductAttention()
query = torch.randn(2, 5, 16, requires_grad=True)
key = torch.randn(2, 7, 16, requires_grad=True)
value = torch.randn(2, 7, 16, requires_grad=True)
y = block(query, key, value)
print(f'SDPA: Q {tuple(query.shape)}, K {tuple(key.shape)}, V {tuple(value.shape)} to {tuple(y.shape)}')
loss = y.square().mean()
inputs = (query, key, value)
loss.backward()
print('Backward done')


SDPA: Q (2, 5, 16), K (2, 7, 16), V (2, 7, 16) to (2, 5, 16)
Backward done


## Multihead Attention (0.1 балл)

![MultiheadAttention](assets/MultiheadAttention.webp)

https://arxiv.org/abs/1706.03762


In [14]:
class MultiHeadAttention(nn.Module):
    def __init__(self, dim: int, num_heads: int):
        super().__init__()

        self.num_heads = num_heads
        self.head_dim = dim // num_heads
        self.q_proj = nn.Linear(dim, dim)
        self.k_proj = nn.Linear(dim, dim)
        self.v_proj = nn.Linear(dim, dim)
        self.out_proj = nn.Linear(dim, dim)
        self.attention = ScaledDotProductAttention()

    def split_heads(self, x: torch.Tensor) -> torch.Tensor:
        batch, length, _ = x.shape
        return x.reshape(batch, length, self.num_heads, self.head_dim).transpose(1, 2)

    def forward(self, query: torch.Tensor, key: torch.Tensor,
                value: torch.Tensor) -> torch.Tensor:
        q = self.split_heads(self.q_proj(query))
        k = self.split_heads(self.k_proj(key))
        v = self.split_heads(self.v_proj(value))
        context = self.attention(q, k, v)
        context = context.transpose(1, 2).reshape(query.shape[0], query.shape[1], -1)
        return self.out_proj(context)


block = MultiHeadAttention(dim=16, num_heads=4)
query = torch.randn(2, 5, 16, requires_grad=True)
key = torch.randn(2, 7, 16, requires_grad=True)
value = torch.randn(2, 7, 16, requires_grad=True)
y = block(query, key, value)
print(f'MHA: Q {tuple(query.shape)}, K {tuple(key.shape)}, V {tuple(value.shape)} to {tuple(y.shape)}')
loss = y.square().mean()
inputs = (query, key, value)
loss.backward()
print('Backward done')


MHA: Q (2, 5, 16), K (2, 7, 16), V (2, 7, 16) to (2, 5, 16)
Backward done


## Transformer Encoder Layer (0.1 балл)


![Transformer Encoder Layer](assets/TransformerEncoder.png)


https://arxiv.org/abs/1706.03762

In [16]:
class TransformerEncoderLayer(nn.Module):
    def __init__(self, dim: int, num_heads: int,
                 mlp_dim: int = None, dropout: float = 0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attention = MultiHeadAttention(dim, num_heads)
        self.norm2 = nn.LayerNorm(dim)
        self.mlp = nn.Sequential(
            nn.Linear(dim, mlp_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(mlp_dim, dim),
            nn.Dropout(dropout),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        normalized = self.norm1(x)
        x = x + self.attention(normalized, normalized, normalized)
        return x + self.mlp(self.norm2(x))


block = TransformerEncoderLayer(dim=16, num_heads=4, mlp_dim = 4*4)
x = torch.randn(2, 7, 16, requires_grad=True)
y = block(x)
print(f'TransformerEncoderLayer: {tuple(x.shape)} -> {tuple(y.shape)}')
loss = y.square().mean()
inputs = (x,)
loss.backward()
print('Backward done')


TransformerEncoderLayer: (2, 7, 16) -> (2, 7, 16)
Backward done


## MLP Mixer (0.1 балл)


![MLPMixer](assets/MLPMixer.png)


https://arxiv.org/abs/2105.01601

In [17]:
class MLPMixerBlock(nn.Module):
    def __init__(self, num_tokens: int, dim: int,
                 token_mlp_dim: int, channel_mlp_dim: int):
        super().__init__()

        self.norm1 = nn.LayerNorm(dim)
        self.token_mlp = nn.Sequential(
            nn.Linear(num_tokens, token_mlp_dim),
            nn.GELU(),
            nn.Linear(token_mlp_dim, num_tokens),
        )
        self.norm2 = nn.LayerNorm(dim)
        self.channel_mlp = nn.Sequential(
            nn.Linear(dim, channel_mlp_dim),
            nn.GELU(),
            nn.Linear(channel_mlp_dim, dim),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        tokens = self.norm1(x).transpose(1, 2)
        x = x + self.token_mlp(tokens).transpose(1, 2)
        return x + self.channel_mlp(self.norm2(x))


block = MLPMixerBlock(num_tokens=7, dim=16, token_mlp_dim=12, channel_mlp_dim=32)
x = torch.randn(2, 7, 16, requires_grad=True)
y = block(x)
print(f'MLPMixerBlock: {tuple(x.shape)} -> {tuple(y.shape)}')
loss = y.square().mean()
inputs = (x,)
loss.backward()
print('Backward done')


MLPMixerBlock: (2, 7, 16) -> (2, 7, 16)
Backward done


## ConvMixer (0.1 балл)

![ConvMixer](assets/ConvMixer.png)


https://arxiv.org/abs/2201.09792

In [18]:
class ConvMixer(nn.Module):

    def __init__(self, channels: int, kernel_size: int = 3):
        super().__init__()

        self.depthwise = nn.Sequential(
            nn.Conv2d(channels, channels, kernel_size=kernel_size,
                      padding=kernel_size // 2, groups=channels),
            nn.ReLU(),
            nn.BatchNorm2d(channels),
        )
        self.pointwise = nn.Sequential(
            nn.Conv2d(channels, channels, kernel_size=1),
            nn.ReLU(),
            nn.BatchNorm2d(channels),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x + self.depthwise(x)
        return self.pointwise(x)


block = ConvMixer(channels=8, kernel_size=5)
x = torch.randn(2, 8, 16, 16, requires_grad=True)
y = block(x)
print(f'ConvMixer: {tuple(x.shape)} to {tuple(y.shape)}')
loss = y.square().mean()
inputs = (x,)
loss.backward()
print('Backward done')


ConvMixer: (2, 8, 16, 16) to (2, 8, 16, 16)
Backward done


## Вопрос (0.2 балла)

Объясните, почему MLPMixer, ConvMixer может работать почти так же эффективно, как обычный Multihead Attention.

Напишите формулу, связывающую Multihead Attention, ConvMixer и MLPMixer

Опишите преимущества и недостатки между ConvMixer, MLPMixer и Multihead Attention

---

Ответ:

Все три механизма позволяют обмениваться информацией между токенами и между каналами признаков.
- MLP-Mixer смешивает токены и каналы с помощью MLP
- ConvMixer — с помощью "depthwise" и "pointwise" сверток

Оба механизма также имеют skip-connections
При подходящем обучении они могут давать сопоставимое качество классификации.

Общую идею можно записать коротко:

$$Y = Mix(C(X)),$$

если берем за C - обмен информацией между токенами, а Mix - смешивание каналов признаков внутри токена

- **Multihead Attention:** связи между токенами вычисляются по входным данным; линейные слои преобразуют каналы до и после attention.
- **MLP-Mixer:** для смешивания токенов и каналов используются два MLP.
- **ConvMixer:** depthwise-свёртка смешивает соседние позиции, а свёртка $1 \times 1$ — каналы.

| Метод | Плюсы | Минусы |
|---|---|---|
| Multihead Attention | Видит все токены сразу и подбирает связи по содержимому входа. | Вычисления становятся дорогими при большом числе токенов (рост квадратично по числу токенов) |
| MLP-Mixer | Простая структура, смешивает информацию между всеми токенами. Рост по числу токенов линейный | Привязан к заданному числу токенов. веса смешивания не вычисляются из входа. Но при увеличении D_T пропорционально числу токенов, тоже может быть дорого|
| ConvMixer | Хорошо учитывает соседство пикселей и может работать с разными размерами изображений. | Для связи далёких участков нужны большие ядра или несколько слоёв. Рецептивное поле локальное и контекст появляется с бОльшей глубиной|
